In [1]:
# Import necessary libraries
import cv2
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_curve, auc, classification_report
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import random
import joblib
from sklearn.svm import SVC
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from scipy.stats import expon
from sklearn.decomposition import PCA
from scipy.stats import randint, uniform
import numpy as np

In [2]:
# Load extracted features
X_train = np.load('../datasets/dataset1/X_train_features.npy')
y_train = np.load('../datasets/dataset1/y_train.npy')
X_test = np.load('../datasets/dataset1/X_test_features.npy')
y_test = np.load('../datasets/dataset1/y_test.npy')

# SVM

In [11]:
# Définir la pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True))
])

# Définir la distribution des hyperparamètres
param_dist = {
    'svm__C': expon(scale=100),  # Distribution exponentielle pour C
    'svm__gamma': expon(scale=.1),  # Distribution exponentielle pour gamma
    'svm__kernel': ['rbf', 'linear']  # Choix entre deux kernels
}

# Configuration de RandomizedSearchCV
random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=10, refit=True, verbose=2, cv=5, n_jobs=-1)

# Entraîner le modèle avec RandomizedSearchCV
random_search.fit(X_train, y_train) #y_train.ravel()

# Afficher les meilleurs paramètres
print(f'Best Parameters: {random_search.best_params_}')

# Sauvegarder le meilleur modèle
best_model_path = '../models/best_svm_model.pkl'
joblib.dump(random_search.best_estimator_, best_model_path)

print(f'Best model saved to {best_model_path}')

Fitting 5 folds for each of 10 candidates, totalling 50 fits


c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Best Parameters: {'svm__C': 101.31862621509056, 'svm__gamma': 0.012283017743719801, 'svm__kernel': 'linear'}
Best model saved to ../models/best_svm_model.pkl


In [12]:
y_pred = random_search.best_estimator_.predict(X_test)
y_proba = random_search.best_estimator_.predict_proba(X_test)[:, 1]

In [13]:
# Calcul des métriques d'évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Affichage des métriques
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1-Score: {f1:.2f}')

Accuracy: 0.87
Precision: 0.82
Recall: 0.94
F1-Score: 0.88


In [9]:
report = classification_report(y_test, y_pred)

print("Classification Report on Test Data:\n", report)

Classification Report on Test Data:
               precision    recall  f1-score   support

         0.0       0.93      0.81      0.87        68
         1.0       0.82      0.94      0.88        64

    accuracy                           0.87       132
   macro avg       0.88      0.87      0.87       132
weighted avg       0.88      0.87      0.87       132



# KNN

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint
import joblib

# Define the pipeline with PCA
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=20)),  # Setting a default number of components to reduce the initial computation
    ('knn', KNeighborsClassifier())
])

# Define the hyperparameter distribution with reduced ranges
param_dist = {
    'pca__n_components': randint(15, 30),  # Narrowed range for PCA components
    'knn__n_neighbors': randint(3, 20),  # Narrowed range for number of neighbors
    'knn__weights': ['uniform', 'distance'],  # Choice of weights
    'knn__p': [1, 2],  # Choice between Manhattan and Euclidean distance
    'knn__leaf_size': randint(20, 40),  # Reduced range for leaf size
    'knn__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']  # Algorithm to use
}

# Configuration of RandomizedSearchCV with reduced iterations and CV folds
n_iter_search = 50  # Reduced number of iterations
random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=n_iter_search, refit=True, verbose=2, cv=5, n_jobs=-1)

# Train the model with RandomizedSearchCV
random_search.fit(X_train, y_train)

# Print the best parameters
print(f'Best Parameters: {random_search.best_params_}')

# Save the best model
best_model_path = '../models/best_knn_model.pkl'
joblib.dump(random_search.best_estimator_, best_model_path)

print(f'Best model saved to {best_model_path}')


Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'knn__algorithm': 'brute', 'knn__leaf_size': 25, 'knn__n_neighbors': 17, 'knn__p': 1, 'knn__weights': 'distance', 'pca__n_components': 29}
Best model saved to ../models/best_knn_model.pkl


c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\neighbors\_classification.py:238: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


In [20]:
y_pred = random_search.best_estimator_.predict(X_test)

In [21]:
# Calcul des métriques d'évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Affichage des métriques
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1-Score: {f1:.2f}')

Accuracy: 0.85
Precision: 0.80
Recall: 0.92
F1-Score: 0.86


In [22]:
report = classification_report(y_test, y_pred)

print("Classification Report on Test Data:\n", report)

Classification Report on Test Data:
               precision    recall  f1-score   support

         0.0       0.91      0.78      0.84        68
         1.0       0.80      0.92      0.86        64

    accuracy                           0.85       132
   macro avg       0.86      0.85      0.85       132
weighted avg       0.86      0.85      0.85       132



# Random Forest

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import RandomizedSearchCV
import joblib
from scipy.stats import randint, uniform

# Define the pipeline with PCA
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=30)),  # Use PCA for dimensionality reduction
    ('rf', RandomForestClassifier())
])

# Define the distribution of hyperparameters
param_dist = {
    'pca__n_components': randint(20, 50),  # Adjust range for PCA components
    'rf__n_estimators': [100, 200, 300, 400, 500],  # Add more options for number of trees
    'rf__max_depth': [None, 10, 20, 30, 40, 50],  # Add more depth options
    'rf__min_samples_split': randint(2, 11),  # Expand range for min_samples_split
    'rf__min_samples_leaf': randint(1, 5),  # Expand range for min_samples_leaf
    'rf__bootstrap': [True, False],  # Use bootstrap samples
    'rf__max_features': ['auto', 'sqrt', 'log2'],  # Add options for max_features
    'rf__criterion': ['gini', 'entropy']  # Criterion for splitting
}

# Configuration of RandomizedSearchCV
random_search = RandomizedSearchCV(pipeline, param_distributions=param_dist, n_iter=100, refit=True, verbose=2, cv=5, n_jobs=-1, random_state=42)

# Train the model with RandomizedSearchCV
random_search.fit(X_train, y_train.ravel())

# Print the best parameters
print(f'Best Parameters: {random_search.best_params_}')

# Save the best model
best_model_path = '../models/best_rf_model.pkl'
joblib.dump(random_search.best_estimator_, best_model_path)

print(f'Best model saved to {best_model_path}')


Fitting 5 folds for each of 100 candidates, totalling 500 fits


c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
175 fits failed out of a total of 500.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
84 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\pipeline.p

Best Parameters: {'pca__n_components': 42, 'rf__bootstrap': True, 'rf__criterion': 'entropy', 'rf__max_depth': 10, 'rf__max_features': 'log2', 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 7, 'rf__n_estimators': 100}
Best model saved to ../models/best_rf_model.pkl


In [4]:
y_pred = random_search.best_estimator_.predict(X_test)

In [5]:
# Calcul des métriques d'évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Affichage des métriques
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1-Score: {f1:.2f}')

Accuracy: 0.88
Precision: 0.81
Recall: 0.98
F1-Score: 0.89


# Decision Tree

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import joblib

# Définir la pipeline avec PCA
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=30)),  # Utiliser PCA pour la réduction de dimension
    ('dt', DecisionTreeClassifier())
])

# Définir la grille des hyperparamètres
param_grid = {
    'dt__max_depth': [None, 10, 20, 30, 40, 50],
    'dt__min_samples_split': [2, 5, 10, 20],
    'dt__min_samples_leaf': [1, 2, 4, 8],
    'dt__criterion': ['gini', 'entropy']
}

# Configuration de GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid=param_grid, refit=True, verbose=2, cv=5, n_jobs=-1)

# Entraîner le modèle avec GridSearchCV
grid_search.fit(X_train, y_train)

# Afficher les meilleurs paramètres
print(f'Best Parameters: {grid_search.best_params_}')

# Sauvegarder le meilleur modèle
best_model_path = '../models/best_dt_model.pkl'
joblib.dump(grid_search.best_estimator_, best_model_path)

print(f'Best model saved to {best_model_path}')


Fitting 5 folds for each of 192 candidates, totalling 960 fits
Best Parameters: {'dt__criterion': 'gini', 'dt__max_depth': None, 'dt__min_samples_leaf': 4, 'dt__min_samples_split': 10}
Best model saved to ../models/best_dt_model.pkl


In [17]:
y_pred = grid_search.best_estimator_.predict(X_test)

In [18]:
# Calcul des métriques d'évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Affichage des métriques
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1-Score: {f1:.2f}')

Accuracy: 0.75
Precision: 0.72
Recall: 0.78
F1-Score: 0.75


# logistic regression

In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import joblib

# Définir la pipeline avec PCA
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),  # Laisser n_components ouvert pour GridSearchCV
    ('logreg', LogisticRegression(max_iter=1000))
])

# Définir la grille des hyperparamètres
param_grid = {
    'pca__n_components': [5, 10, 15, 20, 30],  # Différents nombres de composants pour PCA
    'logreg__C': [0.01, 0.1, 1, 10, 100],  # Paramètre de régularisation
    'logreg__penalty': ['l1', 'l2', 'elasticnet', 'none'],  # Type de régularisation
    'logreg__solver': ['lbfgs', 'liblinear', 'saga', 'newton-cg'],  # Solvers disponibles pour LogisticRegression
    'logreg__l1_ratio': [0, 0.5, 1]  # Spécifique à elasticnet, ignorez pour les autres
}

# Configuration de GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid=param_grid, refit=True, verbose=2, cv=5, n_jobs=-1)

# Entraîner le modèle avec GridSearchCV
grid_search.fit(X_train, y_train)

# Afficher les meilleurs paramètres
print(f'Best Parameters: {grid_search.best_params_}')

# Sauvegarder le meilleur modèle
best_model_path = '../models/best_logreg_model.pkl'
joblib.dump(grid_search.best_estimator_, best_model_path)

print(f'Best model saved to {best_model_path}')


Fitting 5 folds for each of 1200 candidates, totalling 6000 fits


c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\model_selection\_validation.py:540: FitFailedWarning: 
3375 fits failed out of a total of 6000.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
375 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\model_selection\_validation.py", line 888, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\base.py", line 1473, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\pipelin

Best Parameters: {'logreg__C': 1, 'logreg__l1_ratio': 0, 'logreg__penalty': 'l2', 'logreg__solver': 'newton-cg', 'pca__n_components': 20}
Best model saved to ../models/best_logreg_model.pkl


c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1197: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l2)
  warnings.warn(
c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [5]:
y_pred = grid_search.best_estimator_.predict(X_test)

In [6]:
# Calcul des métriques d'évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Affichage des métriques
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1-Score: {f1:.2f}')

Accuracy: 0.73
Precision: 0.68
Recall: 0.86
F1-Score: 0.76


# Naive Bayes

In [8]:
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
import joblib
import numpy as np

# Définir la pipeline avec PCA
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),  # Laisser n_components ouvert pour GridSearchCV
    ('nb', GaussianNB())
])

# Définir la grille des hyperparamètres
param_grid = {
    'pca__n_components': [5, 10, 15, 20, 30],  # Différents nombres de composants pour PCA
    'nb__var_smoothing': np.logspace(-9, 0, 100)  # Recherche sur une échelle logarithmique pour var_smoothing
}

# Configuration de GridSearchCV
grid_search = GridSearchCV(pipeline, param_grid=param_grid, refit=True, verbose=2, cv=5, n_jobs=-1)

# Entraîner le modèle avec GridSearchCV
grid_search.fit(X_train, y_train)

# Afficher les meilleurs paramètres
print(f'Best Parameters: {grid_search.best_params_}')

# Sauvegarder le meilleur modèle
best_model_path = '../models/best_nb_model.pkl'
joblib.dump(grid_search.best_estimator_, best_model_path)

print(f'Best model saved to {best_model_path}')


Fitting 5 folds for each of 500 candidates, totalling 2500 fits
Best Parameters: {'nb__var_smoothing': 0.43287612810830617, 'pca__n_components': 5}
Best model saved to ../models/best_nb_model.pkl


c:\Users\HP\Desktop\sign_verif\myenv\Lib\site-packages\sklearn\utils\validation.py:1339: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [10]:
y_pred = grid_search.best_estimator_.predict(X_test)

In [11]:
# Calcul des métriques d'évaluation
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# Affichage des métriques
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1-Score: {f1:.2f}')

Accuracy: 0.58
Precision: 0.76
Recall: 0.20
F1-Score: 0.32
